In [2]:
import pandas as pd

In [3]:
from pathlib import Path
import pdfplumber

pdf_dir = Path('data') if Path('data').exists() else Path('.')
pdf_files = sorted(pdf_dir.glob('NMOR*.pdf'))

daily_columns = [
    'Date(Nepali)', 'Date(English)', 'Energy_generation_NEA', 'Energy_generation_NEA Subsidiary',
    'Energy_generation_IPP', 'Energy_generation_Import', 'Energy_generation_Total Energy Available',
    'Energy Export', 'Net Energy Met within the country (INPS Demand)', 'Energy Interruption',
    'Energy not served/Generation Deficit', 'Energy Requirement']

demand_columns = [
    'Date (Nepali)', 'Date (English)', 'Peak Time', 'Generation', 'Import',
    'Recorded peak Availability', 'Export', 'Demand at Peak Time', 'Interruption',
    'Deficit', 'Peak Demand met', 'Net value of exchange with India']

daily_rows, demand_rows = [], []

for pdf_path in pdf_files:
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[:10]:
            for table in page.extract_tables() or []:
                for row in table:
                    if not row or len(row) < 12:
                        continue
                    
                    date = (row[0] or '').strip()
                    if '/' not in date or len(date.replace('/', '').replace(' ', '')) != 8:
                        continue
                    
                    if date.count('/') == 1:
                        date = date[:2] + '/' + date[3:5] + '/' + date[5:]
                    
                    third = (row[2] or '').strip()
                    row[0] = date
                    
                    if ':' in third:
                        demand_rows.append(row[:12])
                    elif third.replace(',', '').replace('-', '').isdigit():
                        daily_rows.append(row[:12])

data_energy_value = pd.DataFrame(daily_rows, columns=daily_columns)
demand_and_cross_border_exchange = pd.DataFrame(demand_rows, columns=demand_columns)

In [9]:
data_energy_value.head()

,Date(Nepali),Date(English),Energy_generation_NEA,Energy_generation_NEA Subsidiary,Energy_generation_IPP,Energy_generation_Import,Energy_generation_Total Energy Available,Energy Export,Net Energy Met within the country (INPS Demand),Energy Interruption,Energy not served/Generation Deficit,Energy Requirement
0,01/04/2079,17/07/2022,10382,12923,19249,1313,43867,6834,37034,0,0,37034
1,02/04/2079,18/07/2022,10520,12923,19243,1409,44094,6405,37689,0,0,37689
2,03/04/2079,19/07/2022,10376,12879,19111,1410,43776,7049,36727,20,0,36747
3,04/04/2079,20/07/2022,9389,12892,18652,971,41904,7223,34681,130,0,34811
4,05/04/2079,21/07/2022,9744,12914,18187,715,41559,7598,33962,0,0,33962


In [8]:
demand_and_cross_border_exchange.head()

,Date (Nepali),Date (English),Peak Time,Generation,Import,Recorded peak Availability,Export,Demand at Peak Time,Interruption,Deficit,Peak Demand met,Net value of exchange with India
0,01/04/2079,17/07/2022,19:45,1838,55,1893,214,1679,0,0,1679,-159
1,02/04/2079,18/07/2022,19:40,1828,64,1892,195,1697,0,0,1697,-131
2,03/04/2079,19/07/2022,19:50,1843,80,1923,208,1715,0,0,1715,-128
3,04/04/2079,20/07/2022,19:35,1749,40,1789,178,1611,0,0,1611,-138
4,05/04/2079,21/07/2022,19:40,1791,27,1818,230,1588,0,0,1588,-203
